In [2]:
import pandas as pd

TIMEOUT_VAL = 300
THRESHOLD = 299.9

df = pd.read_csv('results.csv')

solvers = {
    'Proposed SAT': ('SAT', 'SAT TIME (s)'),
    'Basic SAT':    ('BSAT', 'BSAT TIME (s)'),
    'CPLEX MP':     ('MP', 'MP TIME (s)'),
    'CPLEX CP':     ('CP', 'CP TIME (s)'),
    'Gurobi':       ('GUROBI', 'GUROBI TIME (s)'),
}

# Replace TIMEOUT string with 300, convert to numeric
for label, (val_col, time_col) in solvers.items():
    df[time_col] = df[time_col].replace('TIMEOUT', TIMEOUT_VAL)
    df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
    df[val_col]  = df[val_col].replace('-', None)
    df[val_col]  = pd.to_numeric(df[val_col], errors='coerce')

# Extract n from filename e.g. "10_05_005_100_25_1.GSP" -> 10
df['n'] = df['FILENAME'].str.split('_').str[0].astype(int)

ns = sorted(df['n'].unique())

# -------------------------------------------------------
# 1) Flat time lists per solver (for cactus plot)
# -------------------------------------------------------
times = {}
for label, (val_col, time_col) in solvers.items():
    times[label] = df[time_col].dropna().tolist()

# -------------------------------------------------------
# 2) Instances solved per n per solver (for bar chart)
# -------------------------------------------------------
solved = {label: [] for label in solvers}
total_instances = []

for n in ns:
    subset = df[df['n'] == n]
    total_instances.append(len(subset))
    for label, (val_col, time_col) in solvers.items():
        count = (subset[time_col] < THRESHOLD).sum()
        solved[label].append(count)

# -------------------------------------------------------
# Sanity check
# -------------------------------------------------------
print("ns:", ns)
print("total_instances:", total_instances)
for label in solvers:
    print(f"{label} solved: {solved[label]}")

ns: [10, 20, 30, 40, 50]
total_instances: [66, 72, 72, 72, 72]
Proposed SAT solved: [66, 72, 71, 61, 53]
Basic SAT solved: [66, 69, 52, 43, 34]
CPLEX MP solved: [66, 71, 50, 33, 32]
CPLEX CP solved: [66, 72, 72, 72, 72]
Gurobi solved: [66, 72, 50, 35, 33]


In [3]:
df['BSAT']

0        20.0
1        16.0
2        23.0
3        22.0
4        35.0
        ...  
349     696.0
350    1905.0
351       NaN
352       NaN
353     801.0
Name: BSAT, Length: 354, dtype: float64

In [4]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

TIMEOUT = 300
THRESHOLD = 299.9

# -------------------------------------------------------
# Fill in your data here
# -------------------------------------------------------

solved = {
    'Proposed SAT': df['SAT'],
    'Basic SAT':    df['BSAT'],
}

times = {
    'Proposed SAT': df['SAT TIME (s)'],  # flat list of all times across all n
    'Basic SAT':    df['BSAT TIME (s)'],
}

# -------------------------------------------------------
# Styling
# -------------------------------------------------------
colors = {
    'Proposed SAT': '#2196F3',
    'Basic SAT':    '#F44336',
}

markers = {
    'Proposed SAT': 'o',
    'Basic SAT':    's',
}

# -------------------------------------------------------
# Plot 1: Instances Solved vs n (Grouped Bar)
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(ns))
bar_width = 0.35

for idx, (label, counts) in enumerate(solved.items()):
    offset = (idx - len(solved) / 2 + 0.5) * bar_width
    bars = ax.bar(x + offset, counts, bar_width,
                  label=label,
                  color=colors[label],
                  edgecolor='white',
                  linewidth=0.8)
    for bar, val in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                str(val), ha='center', va='bottom',
                fontsize=9, fontweight='bold',
                color=colors[label])

# Total instances reference line
ax.step([x[0] - bar_width, *x, x[-1] + bar_width],
        [total_instances[0], *total_instances, total_instances[-1]],
        where='mid', color='gray', linestyle='--',
        linewidth=1.2, label='Total instances')

ax.set_xlabel('Number of Jobs ($n$)', fontsize=12)
ax.set_ylabel('Instances Solved to Optimality', fontsize=12)
ax.set_title('Instances Solved to Optimality:\nProposed SAT vs Basic SAT Encoding', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels([str(n) for n in ns])
ax.set_ylim(0, max(total_instances) + 10)
ax.legend(fontsize=10)
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('sat_instances_solved.pdf', dpi=300, bbox_inches='tight')
plt.savefig('sat_instances_solved.png', dpi=300, bbox_inches='tight')
plt.close()
print("Plot 1 saved.")

# -------------------------------------------------------
# Plot 2: Cactus Plot (SAT only)
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))

for label, t_list in times.items():
    solved_times = sorted(t for t in t_list if t < THRESHOLD)
    if not solved_times:
        continue
    ax.plot(range(1, len(solved_times) + 1), solved_times,
            label=label,
            color=colors[label],
            marker=markers[label],
            markevery=max(1, len(solved_times) // 15),
            markersize=5,
            linewidth=1.8)

ax.set_xlabel('Number of Instances Solved', fontsize=12)
ax.set_ylabel('Solving Time (seconds)', fontsize=12)
ax.set_title('Cactus Plot: Proposed SAT vs Basic SAT Encoding', fontsize=13)
ax.set_xlim(left=0)
ax.set_ylim(0, TIMEOUT + 10)
ax.axhline(y=TIMEOUT, color='gray', linestyle='--',
           linewidth=1, label='Time limit (300s)')
ax.legend(fontsize=10, loc='upper left')
ax.grid(linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('sat_cactus.pdf', dpi=300, bbox_inches='tight')
plt.savefig('sat_cactus.png', dpi=300, bbox_inches='tight')
plt.close()
print("Plot 2 saved.")

ModuleNotFoundError: No module named 'matplotlib'

In [5]:
import matplotlib
print(matplotlib.__version__
      )

ModuleNotFoundError: No module named 'matplotlib'